# Evaluation Foundations — an honest supervised workflow
Six connected lessons: prediction moment, generalization, classification decisions, probabilities/ranking, uncertainty and workflow provenance. The retained EV01 feasibility section below preserves its original logic. Later sections extend this into one full-module notebook, not a separate mastery credential.
Original fictional embedded data only. No credentials, remote datasets, uploads, installation cells or network access. Use the exact approved local dependency profile below. Actual Google Colab execution remains unverified. All code is trusted author content executed locally, not learner-hosted execution.


# EV01 — Define the prediction moment
ML Mastery — Learn Machine Learning by Doing. Trusted author code, not learner-hosted execution.

This feasibility section belongs to the eventual EV notebook. No estimator is fitted; no credentials, uploads, external data or platform evidence writes. Genuine Colab execution remains unverified. Original fictional rows are not a production documentation-quality dataset.

In [ ]:
import sys, json, random
from importlib.metadata import version
assert sys.version_info[:3] == (3, 13, 16), 'Use the approved local Python profile, or separately verify a new profile.'
pins = {'numpy': '2.5.4', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'scikit-learn': '1.9.1'}
for package, expected in pins.items():
    assert version(package) == expected, f'Unsupported {package} profile'
random.seed(0)
print('Approved local profile asserted; seed=0')

## Problem and visual boundary
At t=0, predict a future house price, cancellation during days1–30, or a human-reviewed label for a submitted guide (Good=0, Needs Improvement=1).

Before t=0: known inputs → t=0: prediction → after t=0: outcome/review. Later outcomes can become training labels, not current features. A correlated post-review flag is unavailable at submission. Predict before running: is an available feature plus a row-overlapping split valid?

In [ ]:
def scenarios():
    specs = [
        ("housing-units", "House price: new independent homes", "housing", False, False),
        ("housing-future", "House price: future sales", "housing", False, True),
        ("churn-customers", "Churn: new customers", "churn", True, False),
        ("churn-future", "Churn: future events for known customers", "churn", False, True),
        ("docs-groups", "Documentation: unseen source/template groups", "docs", True, False),
        ("docs-future", "Documentation: future revisions from unseen families", "docs", True, True),
    ]
    result = []
    for index, (identifier, title, domain, group, time) in enumerate(specs):
        rows = [{"id": f"r{position}", "group": f"g{(position - 1) // 2 + 1}" if group else (f"g{position % 2}" if identifier == "churn-future" else f"g{position}"), "time": position} for position in range(1, 7)]
        plans = [
            {"id": "holdout", "label": "Train r1–r4; test r5–r6", "train": ["r1", "r2", "r3", "r4"], "test": ["r5", "r6"]},
            {"id": "overlap", "label": "Train r1–r4; test r4–r6", "train": ["r1", "r2", "r3", "r4"], "test": ["r4", "r5", "r6"]},
            {"id": "crossed", "label": "Train r1,r3,r5; test r2,r4,r6" if group else "Train r1,r5,r6; test r2,r3,r4", "train": ["r1", "r3", "r5"] if group else ["r1", "r5", "r6"], "test": ["r2", "r4", "r6"] if group else ["r2", "r3", "r4"]},
        ]
        if not group and not time:
            plans[2] = {"id": "crossed", "label": "Train r1–r5; test r5–r6", "train": ["r1", "r2", "r3", "r4", "r5"], "test": ["r5", "r6"]}
        safe, future = {"housing": ("area at listing", "future sale proceeds"), "churn": ("usage before decision", "cancellation after decision"), "docs": ("submitted text at review request", "post-review correction flag")}[domain]
        result.append({"id": identifier, "title": title, "domain": domain, "requireGroup": group, "requireTime": time, "rows": rows, "plans": plans[index % 3:] + plans[:index % 3], "features": [{"id": "available", "label": safe, "observedAt": -1}, {"id": "future", "label": future, "observedAt": 1}]})
    return result

def analyze(scene, feature_id, plan_id):
    feature = next(item for item in scene["features"] if item["id"] == feature_id)
    plan = next(item for item in scene["plans"] if item["id"] == plan_id)
    rows = {row["id"]: row for row in scene["rows"]}
    overlap = sorted(set(plan["train"]) & set(plan["test"]))
    groups = sorted({rows[key]["group"] for key in plan["train"]} & {rows[key]["group"] for key in plan["test"]}) if scene["requireGroup"] else []
    future = [key for key in plan["train"] if rows[key]["time"] >= min(rows[item]["time"] for item in plan["test"])] if scene["requireTime"] else []
    feature_valid = feature["observedAt"] <= 0
    split_valid = not (overlap or groups or future)
    reason = "both-valid" if feature_valid and split_valid else "feature-future" if split_valid else "split-invalid" if feature_valid else "both-invalid"
    return {"featureValid": feature_valid, "splitValid": split_valid, "valid": feature_valid and split_valid, "overlap": overlap, "groups": groups, "future": future, "reason": reason}


In [ ]:
scenes = scenarios()
scene = scenes[0]
result = analyze(scene, 'available', 'overlap')
assert result['overlap'] == ['r4']
assert not result['valid']
print(json.dumps(result, sort_keys=True))

## Mathematics, Python and changed conditions
E(0)={j: observed_at[j]≤0}; T∩H must be empty. Unseen entities additionally require disjoint groups, and future-cohort evaluation requires earlier training times. These necessary conditions do not replace a real label-availability audit.

Guided repair: replace the authored plan ID `overlap` with `holdout`, not arbitrary code. Then change `available` to `future`: a good split does not repair an unavailable feature.

In [ ]:
safe = analyze(scene, 'available', 'holdout')
future_feature = analyze(scene, 'future', 'holdout')
assert safe['valid']
assert not future_feature['valid'] and future_feature['splitValid']
print(json.dumps({'safe': safe['valid'], 'future_feature': future_feature['reason']}, sort_keys=True))

In [ ]:
results = []
for scene in scenes:
    for feature in scene['features']:
        for plan in scene['plans']:
            results.append({'config': scene['id'] + ':' + feature['id'] + ':' + plan['id'], **analyze(scene, feature['id'], plan['id'])})
assert len(results) == 36
assert sum(result['valid'] for result in results) == 6
assert analyze(scenes[2], 'available', 'crossed')['groups'] == ['g1', 'g2', 'g3']
assert analyze(scenes[1], 'available', 'crossed')['future'] == ['r5', 'r6']
assert analyze(scenes[5], 'available', 'crossed')['future'] == ['r3', 'r5']
print(json.dumps({'states': len(results), 'valid': sum(result['valid'] for result in results), 'results': results}, sort_keys=True))

## Evaluate and compare
All36 configurations check information permissions, not fitted performance. Linear Regression produces a numeric price; classification models are candidates for churn/document labels, but no algorithm repairs leakage. Association does not establish that increasing usage or padding words improves an outcome.

Independent transfer: explain why a disjoint random row split might still leak documentation templates; contrast future events of known customers with unseen-customer deployment. No notebook execution grants mastery.

Expected assertions: r4 overlaps; six valid states out of36; group overlap g1/g2/g3; future-training rows r5/r6 for housing and r3/r5 for future documentation. Local two-kernel reproducibility is distinct from genuine Colab acceptance.

## EV02 / EV05 — compare on unseen rows, not training fit
House prices follow a fictional linear relationship. Select the complete preprocessing/estimator workflow using development folds and compare with a training-fitted baseline. Freeze the choice before evaluating the last two houses. This reuses Linear Regression; no Polynomial Regression algorithm is introduced.
Predict: which workflow will beat a constant baseline on these synthetic houses? Why would repeatedly choosing by final-test error invalidate the comparison?


In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
np.random.seed(1729)
houses = pd.DataFrame({"id": [f"H{i:02}" for i in range(1,13)], "area": range(1,13)})
houses["price"] = 3 + 2 * houses["area"]
import hashlib
house_checksum = hashlib.sha256(json.dumps(houses.to_dict(orient="records"), sort_keys=True, separators=(",",":")).encode()).hexdigest()
assert house_checksum == "6befc51fd320313903991e366cb19ca142bb1765ce69564bc7b6b32247634d63"
development, final_test = houses.iloc[:10], houses.iloc[10:]
folds = list(KFold(n_splits=3, shuffle=False).split(development))
workflow = make_pipeline(StandardScaler(), LinearRegression())
baseline = DummyRegressor(strategy="mean")
X, y = development[["area"]], development["price"]
workflow_errors = -cross_val_score(workflow, X, y, cv=folds, scoring="neg_mean_absolute_error")
baseline_errors = -cross_val_score(baseline, X, y, cv=folds, scoring="neg_mean_absolute_error")
assert workflow_errors.mean() < baseline_errors.mean()
for train, validation in folds:
    assert not set(development.iloc[train].id) & set(development.iloc[validation].id)
selected = "linear_pipeline"
workflow.fit(X, y)
final_predictions = workflow.predict(final_test[["area"]])
final_mae = mean_absolute_error(final_test.price, final_predictions)
assert final_mae < 1e-10
print(json.dumps({"selection_source": "development_only", "selected": selected,
 "cv_mae": np.round(workflow_errors,12).tolist(),
 "baseline_cv_mae": np.round(baseline_errors,12).tolist(),
 "final_ids": final_test.id.tolist(), "final_mae": round(float(final_mae),12)}, sort_keys=True))



### Capacity and fixed-resample replay
The fixed train/development MAE pairs (8,9),(3,4),(1,12) illustrate why a more flexible fit need not generalize better. They are not newly fitted polynomial curves. Initial prediction may be wrong: compare and explain the result. A seed gives reproducibility, not certainty.


In [ ]:
capacity = {"low": (8,9), "middle": (3,4), "high": (1,12)}
best = min(capacity, key=lambda name: capacity[name][1])
assert best == "middle"
relative_reduction = (5 - capacity[best][1]) / 5
assert np.isclose(relative_reduction, .2)
fixed_errors = [[4,4,4],[2,4,6],[1,4,7]]
spreads = [float(np.std(values, ddof=0)) for values in fixed_errors]
assert all(np.mean(values) == 4 for values in fixed_errors)
assert np.isclose(spreads[1], np.sqrt(8/3))
assert r2_score([1,3],[3,3],force_finite=False) == -1
print(json.dumps({"capacity": capacity, "development_best": best, "relative_reduction": relative_reduction,
 "descriptive_spreads": spreads, "confidence_interval": "not_estimated",
 "zero_baseline_reduction": {"tag":"undefined","reason":"zero-baseline-error"}}, sort_keys=True))


## EV03 / EV04 — decisions, costs, ranking and probability quality
Positive=1 means Needs Improvement, negative=0 means Good. These are fictional labels, not validated production annotations. Predict before changing the threshold: do fitted probabilities change? Does majority accuracy prove detection? What if a class is absent?
For y=[1,0], p=[.8,.2], binary Brier=.04 and natural log loss=−ln(.8). Ranking can be perfect while probability values are poor. Use direct denominators and tagged undefined states. Mathematical endpoint loss is not a numerical clipping approximation.


In [ ]:
import math
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, brier_score_loss, log_loss, roc_auc_score, average_precision_score
fixtures = json.loads("[{\"id\":\"ev-e2.balanced.v1\",\"y\":[0,0,1,1],\"p\":[0.1,0.4,0.6,0.9],\"fp_cost\":1,\"fn_cost\":1},{\"id\":\"ev-e2.same-ranking.v1\",\"y\":[0,0,1,1],\"p\":[0.01,0.49,0.51,0.99],\"fp_cost\":1,\"fn_cost\":1},{\"id\":\"ev-e2.imbalanced.v1\",\"y\":[0,0,0,0,0,0,0,1],\"p\":[0.1,0.2,0.3,0.4,0.2,0.1,0.3,0.6],\"fp_cost\":1,\"fn_cost\":5},{\"id\":\"ev-e2.no-positives.v1\",\"y\":[0,0,0,0],\"p\":[0.1,0.2,0.6,0.8],\"fp_cost\":1,\"fn_cost\":5},{\"id\":\"ev-e2.no-negatives.v1\",\"y\":[1,1,1,1],\"p\":[0.1,0.2,0.6,0.8],\"fp_cost\":1,\"fn_cost\":5},{\"id\":\"ev-e2.endpoints.v1\",\"y\":[0,1,0,1],\"p\":[0,1,1,0],\"fp_cost\":1,\"fn_cost\":5},{\"id\":\"ev-e2.all-tied.v1\",\"y\":[0,1,0,1],\"p\":[0.5,0.5,0.5,0.5],\"fp_cost\":1,\"fn_cost\":1},{\"id\":\"ev-e2.reversed-ranking.v1\",\"y\":[0,0,1,1],\"p\":[0.9,0.6,0.4,0.1],\"fp_cost\":1,\"fn_cost\":1}]")
classification_checksum = hashlib.sha256(json.dumps(fixtures, sort_keys=True, separators=(",",":")).encode()).hexdigest()
assert classification_checksum == "30f2de9c976ed080dd6e1aa7128393619a38583f92d8903af5c86515db86875f"
def divide_or_tag(numerator, denominator, reason):
    return numerator / denominator if denominator else {"tag": "undefined", "reason": reason}
def mathematical_loss(y, probabilities):
    terms = [probability if target else 1-probability for target,probability in zip(y,probabilities)]
    return -sum(math.log(value) for value in terms)/len(terms) if all(terms) else {"tag":"infinity","sign":"positive","reason":"zero-probability-observed-outcome"}
classified = []
for fixture in fixtures:
    y, probabilities = fixture["y"], fixture["p"]
    for threshold in [0,.5,.75,1]:
        predicted = [int(probability >= threshold) for probability in probabilities]
        tn,fp,fn,tp = [int(value) for value in confusion_matrix(y,predicted,labels=[0,1]).ravel()]
        result = {"fixture":fixture["id"],"threshold":threshold,"tp":tp,"fp":fp,"tn":tn,"fn":fn,
          "precision":divide_or_tag(tp,tp+fp,"no-predicted-positives"),
          "recall":divide_or_tag(tp,tp+fn,"no-actual-positives"),
          "f1":divide_or_tag(2*tp,2*tp+fp+fn,"no-actual-or-predicted-positives"),
          "cost":fixture["fp_cost"]*fp+fixture["fn_cost"]*fn,
          "brier":float(brier_score_loss(y,probabilities,pos_label=1,scale_by_half=True)),
          "mathematical_log_loss":mathematical_loss(y,probabilities),
          "library_clipped_log_loss":float(log_loss(y,np.array(probabilities,dtype=np.float64),labels=[0,1]))}
        classified.append(result)
assert len(classified) == 32
majority = next(row for row in classified if row["fixture"]=="ev-e2.imbalanced.v1" and row["threshold"]==.75)
assert majority["recall"] == 0 and majority["precision"]["tag"] == "undefined" and majority["f1"] == 0
endpoint = next(row for row in classified if row["fixture"]=="ev-e2.endpoints.v1")
assert endpoint["mathematical_log_loss"]["tag"] == "infinity"
assert math.isfinite(endpoint["library_clipped_log_loss"])
assert [int(value >= .5) for value in [.5,.5]] == [1,1]
for fixture in fixtures:
    rows = [row for row in classified if row["fixture"] == fixture["id"]]
    assert all(row["brier"] == rows[0]["brier"] for row in rows)
print(json.dumps({"configurations":len(classified),"majority_counterexample":majority,"endpoint":endpoint},sort_keys=True))



In [ ]:
first, same_rank = fixtures[0], fixtures[1]
assert roc_auc_score(first["y"],first["p"]) == roc_auc_score(same_rank["y"],same_rank["p"]) == 1
assert average_precision_score(first["y"],first["p"]) == average_precision_score(same_rank["y"],same_rank["p"]) == 1
assert brier_score_loss(first["y"],first["p"],scale_by_half=True) != brier_score_loss(same_rank["y"],same_rank["p"],scale_by_half=True)
bins = []
for lower,upper in [(0,.5),(.5,1)]:
    indexes = [index for index,p in enumerate(first["p"]) if lower <= p and (p < upper if upper < 1 else p <= upper)]
    bins.append({"count":len(indexes),"mean_probability":float(np.mean([first["p"][index] for index in indexes])),
                 "observed_fraction":float(np.mean([first["y"][index] for index in indexes]))})
print(json.dumps({"calibration_diagnostic":bins,"claim":"Equal ranking does not establish calibrated future risk or causation."},sort_keys=True))


## EV06 — fit receipts and honest reports
Predict the centered query before reveal: may validation rows determine the mean? Does fitting once on all training data protect every CV fold? Trace the actual IDs.
Guided repair: choose `fold_train` rather than `global` below. Explain why a pipeline must fit separately within each fold, and why a global score cannot guarantee new-template or rare-subgroup performance.


In [ ]:
values = np.array([2.,4.,6.,8.,20.,22.]).reshape(-1,1)
train_mean = StandardScaler(with_std=False).fit(values[:4]).mean_[0]
leaked_mean = StandardScaler(with_std=False).fit(values).mean_[0]
fold_a = StandardScaler(with_std=False).fit(values[:2]).mean_[0]
fold_b = StandardScaler(with_std=False).fit(values[2:4]).mean_[0]
assert train_mean == 5 and np.isclose(leaked_mean,31/3)
assert fold_a == 3 and fold_b == 7
repair = "fold_train"
assert repair == "fold_train", "Use only the current fold's training IDs, then transform validation."
report = {"dataset":"original-fictional-houses.v1","target":"price","unit":"fictional currency",
 "development_ids":development.id.tolist(),"final_test_ids":final_test.id.tolist(),
 "selection":"development-only; frozen linear pipeline","fold_train_ids":[development.iloc[train].id.tolist() for train,_ in folds],
 "seed":1729,"metric":"MAE; same rows; positive error","population_limit":"No guarantee for new regions or sparse subgroups",
 "colab":"UNVERIFIED: local kernels only","mastery":"Notebook execution never awards platform mastery"}
print(json.dumps({"train_mean":float(train_mean),"leaked_mean":float(leaked_mean),"fold_means":[float(fold_a),float(fold_b)],"report":report},sort_keys=True))
print(json.dumps({"rows":12,"folds":3,"classification_configurations":32,"positive_class":"Needs Improvement=1","final_mae_rounded":round(float(final_mae),12)},sort_keys=True))


## Independent transfer
1. Explain why selecting a threshold on final-test labels leaks even when coefficients stay fixed.
2. Compare undefined precision with recall0 and direct-count F1=0 in the majority counterexample.
3. Explain mathematical infinite log loss versus library clipping, and why a correct certainty contribution is0.
4. State what ROC/AP cannot establish about calibration or causation.
5. Write the permitted IDs for a CV fold's learned transform and a final-test permission record.
Check the counterexamples above, then complete a fresh full independent platform form. Notebook use, practice and experiment credit never confer module mastery.
